In [2]:
import json
import pandas as pd

LOG_PATH = r'C:\Users\zaimr\projects\barberi-pantaucukur\barberi-backend\PantauCukur\core\logs\ai_engine.jsonl'

records = []
errors = 0

with open(LOG_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            errors += 1
            print(f"⚠️ Baris {i} corrupt, di-skip")

df = pd.DataFrame(records)
print(f"✅ Total baris valid: {len(df)} | Corrupt: {errors}")
df.head()

✅ Total baris valid: 70 | Corrupt: 0


,text,record
0,2026-08-13 15:10:20.728 | INFO | system | REDI...,"{'elapsed': {'repr': '0:00:00.078514', 'second..."
1,2026-08-13 15:10:20.730 | INFO | system | ENGI...,"{'elapsed': {'repr': '0:00:00.080513', 'second..."
2,2026-08-13 15:10:20.730 | INFO | system | CONF...,"{'elapsed': {'repr': '0:00:00.080513', 'second..."
3,2026-08-13 15:10:20.731 | INFO | system | MODE...,"{'elapsed': {'repr': '0:00:00.081514', 'second..."
4,2026-08-13 15:10:24.626 | INFO | system | MODE...,"{'elapsed': {'repr': '0:00:03.976501', 'second..."


In [39]:
import json
import pandas as pd

pd.set_option('display.max_colwidth', 500)  # atau None untuk unlimited
pd.set_option('display.max_rows', 500)

LOG_PATH = r'C:\Users\zaimr\projects\barberi-pantaucukur\barberi-backend\PantauCukur\core\services\logs\ai_engine.jsonl'

data = []
with open(LOG_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        log = json.loads(line)
        
        # ✅ AMBIL DARI RECORD (JSON) - MUDAH & KONSISTEN
        row = {
            'timestamp': log['record']['time']['timestamp'],
            'level': log['record']['level']['name'],
            'component': log['record']['extra']['component'],
            'event': log['record']['extra']['event'],
            'file': log['record']['extra']['file'],
            'function': log['record']['extra']['function'],
            'line': log['record']['extra']['line'],
            'elapsed_seconds': log['record']['elapsed']['seconds'],
        }
        
        # 📝 AMBIL KEY-VALUE DARI TEXT (karena di record gak ada)
        text = log['text']
        parts = text.split(' | ')
        for part in parts[4:]:  # skip timestamp, level, component, event
            if '=' in part:
                key, val = part.split('=', 1)
                row[key.strip()] = val.strip()
        
        data.append(row)

df = pd.DataFrame(data)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')

print(df[['datetime', 'level', 'component', 'event', 'chair_id', 'status_code']].head())

                       datetime level component            event chair_id  \
0 2026-08-15 06:30:02.638056040  INFO    system  REDIS_CONNECTED      NaN   
1 2026-08-15 06:30:02.639055967  INFO    system     ENGINE_START      NaN   
2 2026-08-15 06:30:02.640060902  INFO    system    CONFIG_LOADED      NaN   
3 2026-08-15 06:30:02.640060902  INFO    system    MODEL_LOADING      NaN   
4 2026-08-15 06:30:02.694570065  INFO    system     MODEL_LOADED      NaN   

  status_code  
0         NaN  
1         NaN  
2         NaN  
3         NaN  
4         NaN  


In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 44 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   timestamp                   80 non-null     float64       
 1   level                       80 non-null     str           
 2   component                   80 non-null     str           
 3   event                       80 non-null     str           
 4   file                        80 non-null     str           
 5   function                    80 non-null     str           
 6   line                        80 non-null     int64         
 7   elapsed_seconds             80 non-null     float64       
 8   status                      12 non-null     str           
 9   headless_mode               2 non-null      str           
 10  camera_url                  2 non-null      str           
 11  skip_frames                 4 non-null      str           
 12  use_sco

In [49]:
df['line'].shape

(80,)

In [50]:
df.shape

(80, 44)

In [51]:

df['status_code'] = pd.to_numeric(df['status_code'], errors='coerce')

# Baru deh bisa describe
df['status_code'].describe()

count     11.0
mean     404.0
std        0.0
min      404.0
25%      404.0
50%      404.0
75%      404.0
max      404.0
Name: status_code, dtype: float64